# Phase 2 L-NER — Encoder Fine-tuning (BIO + CRF)

Uses the **encoder** from the pretrained encoder-decoder model (~48M encoder params).
Encoder is naturally **bidirectional** — no BiLSTM hack needed.

**Colab + Drive version. Upload best_encdec.pt, bpe_40k.json, a1_tokenizer.py to your Drive.**

In [ ]:
import os, sys, re, json, math, time, random, bisect, copy, zipfile, glob, shutil
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from google.colab import drive
drive.mount('/content/drive')

print("Imports OK")
print("PyTorch:", torch.__version__)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1 — Configuration

In [ ]:
@dataclass
class NERConfig:
    # Model — must match encoder-decoder Phase 1
    vocab_size: int      = 40_000
    d_model: int         = 640
    n_layers: int        = 5       # encoder layers only
    n_heads: int         = 10      # query heads
    n_kv_heads: int      = 5       # KV heads (GQA 2:1)
    d_ff: int            = 1728    # SwiGLU
    max_seq_len: int     = 512
    rms_eps: float       = 1e-5
    rope_theta: float    = 10000.0
    dropout: float       = 0.1

    # NER
    num_labels: int      = 12
    bio_num_tags: int    = 25      # 2*12 + 1 (O)
    window_size: int     = 510
    window_stride: int   = 255

    # Training
    lr: float            = 1e-5    # encoder LR
    min_lr: float        = 1e-7
    warmup_steps: int    = 200
    num_epochs: int      = 25
    batch_size: int      = 8       # encoder is small, fits easily
    grad_accum: int      = 2       # effective batch = 16
    weight_decay: float  = 0.01
    max_grad_norm: float = 1.0
    use_amp: bool        = True
    seed: int            = 42

    def __post_init__(self):
        self.head_dim = self.d_model // self.n_heads

CFG = NERConfig()
OUT       = Path(".")
CKPT_DIR  = OUT / "ner_checkpoints"; CKPT_DIR.mkdir(exist_ok=True)
SUB_DIR   = OUT / "submissions";     SUB_DIR.mkdir(exist_ok=True)

print(f"Config: {CFG.d_model}d x {CFG.n_layers}L encoder, heads={CFG.n_heads}/{CFG.n_kv_heads}")
print(f"Batch={CFG.batch_size} x grad_accum={CFG.grad_accum} = effective {CFG.batch_size * CFG.grad_accum}")

Config: 640d x 5L encoder, heads=10/5
Batch=8 x grad_accum=2 = effective 16


## 2 — Load Tokenizer

In [ ]:
# ── Search Drive for required files ──
def find_in_drive(filename):
    """Search Google Drive for a file by name."""
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        if filename in files:
            return os.path.join(root, filename)
    return None

PHASE1_CKPT = find_in_drive("bestencdec.pt")
tok_json_path = find_in_drive("bpe_40k.json")
tok_py_path = find_in_drive("a1_tokenizer.py")

for f, desc in [(PHASE1_CKPT, "bestencdec.pt"), (tok_json_path, "bpe_40k.json"), (tok_py_path, "a1_tokenizer.py")]:
    assert f is not None and os.path.isfile(f), f"Not found in Drive: {desc}"
    print(f"  Found {desc}: {f}")

# Add tokenizer directory to path
sys.path.insert(0, os.path.dirname(tok_py_path))
from a1_tokenizer import BPETokenizer
tokenizer = BPETokenizer.load(tok_json_path)
CFG.vocab_size = tokenizer.vocab_size
print(f"Tokenizer loaded: vocab_size={tokenizer.vocab_size}")
_t = "The Supreme Court held that"
assert tokenizer.decode(tokenizer.encode(_t)) == _t, "Round-trip failed"
print("Tokenizer OK")

  Found bestencdec.pt: /content/drive/MyDrive/bestencdec.pt
  Found bpe_40k.json: /content/drive/MyDrive/bpe_40k.json
  Found a1_tokenizer.py: /content/drive/MyDrive/a1_tokenizer.py
Tokenizer loaded: vocab_size=40000
Tokenizer OK


## 3 — Download NER Data

In [ ]:
!pip -q install huggingface_hub
from huggingface_hub import hf_hub_download

HF_REPO = "Exploration-Lab/CS-781-Capstone"
HF_TOKEN = ""

NER_FILES = {
    "fold1_text":  "Phase_2_LNER/Train/fold1-text.json",
    "fold1_gold":  "Phase_2_LNER/Train/fold1-gold.json",
    "fold2_text":  "Phase_2_LNER/Train/fold2-text.json",
    "fold2_gold":  "Phase_2_LNER/Train/fold2-gold.json",
    "labels":      "Phase_2_LNER/Train/labels.txt",
    "fold3_text":  "Phase_2_LNER/Test/fold3-text.json",
}

ner_paths = {}
for key, fn in NER_FILES.items():
    ner_paths[key] = hf_hub_download(HF_REPO, fn, repo_type="dataset", token=HF_TOKEN)
    print(f"  {key:12s} -> downloaded")

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

  fold1_text   -> downloaded


fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

  fold1_gold   -> downloaded


fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

  fold2_text   -> downloaded


... (2 outputs left out)

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

  labels       -> downloaded


fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

  fold3_text   -> downloaded


## 4 — Parse Data & BIO Label Maps

In [ ]:
with open(ner_paths["labels"], "r") as f:
    LABELS = [l.strip() for l in f if l.strip()]
print(f"Entity labels ({len(LABELS)}): {LABELS}")

O_TAG = 0
tag2id = {"O": O_TAG}
id2tag = {O_TAG: "O"}
for i, lab in enumerate(LABELS):
    tag2id[f"B-{lab}"] = 2*i + 1
    tag2id[f"I-{lab}"] = 2*i + 2
    id2tag[2*i + 1] = f"B-{lab}"
    id2tag[2*i + 2] = f"I-{lab}"

CFG.num_labels = len(LABELS)
CFG.bio_num_tags = len(tag2id)
print(f"BIO tags: {CFG.bio_num_tags}")

def load_json(p):
    with open(p, "r") as f:
        return json.load(f)

fold1_text = load_json(ner_paths["fold1_text"])
fold1_gold = load_json(ner_paths["fold1_gold"])
fold2_text = load_json(ner_paths["fold2_text"])
fold2_gold = load_json(ner_paths["fold2_gold"])
fold3_text = load_json(ner_paths["fold3_text"])
print(f"Fold 1: {len(fold1_text)}, Fold 2: {len(fold2_text)}, Fold 3 (test): {len(fold3_text)}")

Entity labels (12): ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
BIO tags: 25
Fold 1: 35, Fold 2: 35, Fold 3 (test): 35


## 5 — Offset-Tracking Encoder

In [ ]:
def encode_with_offsets(tokenizer, text):
    token_ids = []
    char_spans = []
    for m in re.finditer(r'\S+', text):
        word = m.group()
        ws = m.start()
        char_byte_bounds = [0]
        for ch in word:
            char_byte_bounds.append(char_byte_bounds[-1] + len(ch.encode('utf-8')))
        bpe_syms = tokenizer._bpe(word)
        SPACE_BYTES = 1
        fb_pos = 0
        for sym in bpe_syms:
            tok_id = tokenizer.token_to_id.get(sym, tokenizer.token_to_id.get('<unk>', 1))
            n_bytes = len(sym)
            fb_end = fb_pos + n_bytes
            wb_s = max(0, fb_pos - SPACE_BYTES)
            wb_e = max(0, fb_end - SPACE_BYTES)
            wb_e = min(wb_e, char_byte_bounds[-1])
            wb_s = min(wb_s, char_byte_bounds[-1])
            if wb_e > wb_s:
                c_s = bisect.bisect_right(char_byte_bounds, wb_s) - 1
                c_e = bisect.bisect_left(char_byte_bounds, wb_e)
                c_s = max(0, c_s)
                c_e = min(len(word), c_e)
                char_spans.append((ws + c_s, ws + c_e))
            else:
                char_spans.append((ws, ws))
            token_ids.append(tok_id)
            fb_pos = fb_end
    return token_ids, char_spans

def get_doc_text(doc_entry):
    if isinstance(doc_entry, str):
        return doc_entry
    if isinstance(doc_entry, dict):
        return doc_entry.get("text", "")
    return str(doc_entry)

print("Offset encoder defined")

Offset encoder defined


## 6 — Encoder Model Architecture (matches Phase 1 enc-dec)

In [ ]:
# ── RMSNorm (matches enc-dec: uses self.g) ──
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.g = nn.Parameter(torch.ones(d))

    def forward(self, x):
        return self.g * x * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)

# ── RoPE (INTERLEAVED — matches enc-dec, different from decoder model!) ──
def build_rope_cache(seq_len, head_dim, theta=10000.0, device='cpu'):
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(seq_len, device=device).float()
    freqs = torch.outer(t, inv_freq)
    return torch.cos(freqs), torch.sin(freqs)

def apply_rope(x, cos, sin):
    T = x.size(-2)
    cos = cos[:T].unsqueeze(0).unsqueeze(0)
    sin = sin[:T].unsqueeze(0).unsqueeze(0)
    x1 = x[..., 0::2]
    x2 = x[..., 1::2]
    r1 = x1 * cos - x2 * sin
    r2 = x1 * sin + x2 * cos
    return torch.stack([r1, r2], dim=-1).flatten(-2)

# ── GQA Self-Attention (bidirectional for encoder, with optional padding mask) ──
class GQASelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_q = cfg.n_heads
        self.n_kv = cfg.n_kv_heads
        self.head_dim = cfg.head_dim
        self.n_rep = self.n_q // self.n_kv
        self.wq = nn.Linear(cfg.d_model, self.n_q * self.head_dim, bias=False)
        self.wk = nn.Linear(cfg.d_model, self.n_kv * self.head_dim, bias=False)
        self.wv = nn.Linear(cfg.d_model, self.n_kv * self.head_dim, bias=False)
        self.wo = nn.Linear(self.n_q * self.head_dim, cfg.d_model, bias=False)

    def forward(self, x, cos, sin, attn_mask=None):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.n_q,  self.head_dim).transpose(1, 2)
        k = self.wk(x).view(B, T, self.n_kv, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(B, T, self.n_kv, self.head_dim).transpose(1, 2)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)
        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)
        # Bidirectional attention (is_causal=False)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, is_causal=False)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.wo(out)

# ── SwiGLU FFN ──
class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d_model, bias=False)
        self.w3 = nn.Linear(d_model, d_ff, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

# ── Encoder Block (matches enc-dec naming: n1, attn, n2, ff) ──
class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n1 = RMSNorm(cfg.d_model, cfg.rms_eps)
        self.attn = GQASelfAttention(cfg)
        self.n2 = RMSNorm(cfg.d_model, cfg.rms_eps)
        self.ff = SwiGLU(cfg.d_model, cfg.d_ff)

    def forward(self, x, cos, sin, attn_mask=None):
        x = x + self.attn(self.n1(x), cos, sin, attn_mask)
        x = x + self.ff(self.n2(x))
        return x

# ── Standalone Encoder (param names match enc-dec checkpoint) ──
class LegalEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.src_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.enc_blocks = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.enc_norm = RMSNorm(cfg.d_model, cfg.rms_eps)
        max_len = cfg.max_seq_len * 4
        cos, sin = build_rope_cache(max_len, cfg.head_dim, cfg.rope_theta)
        self.register_buffer('rope_cos', cos, persistent=False)
        self.register_buffer('rope_sin', sin, persistent=False)

    def forward(self, ids, attention_mask=None):
        B, T = ids.shape
        x = self.src_emb(ids)
        # Build padding mask for SDPA: (B, 1, 1, T) additive mask
        amask = None
        if attention_mask is not None:
            amask = torch.zeros(B, 1, 1, T, device=ids.device, dtype=x.dtype)
            amask.masked_fill_(attention_mask[:, None, None, :] == 0, float("-inf"))
        for blk in self.enc_blocks:
            x = blk(x, self.rope_cos, self.rope_sin, amask)
        return self.enc_norm(x)

print("Encoder architecture defined")
n_enc_params = sum(p.numel() for p in LegalEncoder(CFG).parameters())
print(f"Encoder params: {n_enc_params/1e6:.1f}M")

Encoder architecture defined
Encoder params: 48.3M


## 7 — CRF (bug-fixed) + NER Model

In [ ]:
class CRF(nn.Module):
    def __init__(self, num_tags):
        super().__init__()
        self.num_tags = num_tags
        self.transitions = nn.Parameter(torch.randn(num_tags, num_tags))
        self.start_transitions = nn.Parameter(torch.randn(num_tags))
        self.end_transitions = nn.Parameter(torch.randn(num_tags))
        nn.init.uniform_(self.transitions, -0.1, 0.1)
        nn.init.uniform_(self.start_transitions, -0.1, 0.1)
        nn.init.uniform_(self.end_transitions, -0.1, 0.1)

    def forward(self, emissions, tags, mask):
        gold_score = self._score_sentence(emissions, tags, mask)
        forward_score = self._forward_alg(emissions, mask)
        return (forward_score - gold_score).mean()

    def _score_sentence(self, emissions, tags, mask):
        B, T, C = emissions.shape
        score = self.start_transitions[tags[:, 0]] + emissions[:, 0].gather(1, tags[:, 0:1]).squeeze(1)
        for t in range(1, T):
            trans = self.transitions[tags[:, t], tags[:, t-1]]
            emit = emissions[:, t].gather(1, tags[:, t:t+1]).squeeze(1)
            step_score = trans + emit
            score = torch.where(mask[:, t], score + step_score, score)
        last_idx = mask.long().sum(dim=1) - 1
        last_tags = tags.gather(1, last_idx.unsqueeze(1)).squeeze(1)
        score = score + self.end_transitions[last_tags]
        return score

    def _forward_alg(self, emissions, mask):
        B, T, C = emissions.shape
        alpha = self.start_transitions.unsqueeze(0) + emissions[:, 0]
        for t in range(1, T):
            emit = emissions[:, t].unsqueeze(2)   # FIXED: was unsqueeze(1)
            trans = self.transitions.unsqueeze(0)
            alpha_exp = alpha.unsqueeze(1) + trans + emit
            new_alpha = torch.logsumexp(alpha_exp, dim=2)
            alpha = torch.where(mask[:, t].unsqueeze(1), new_alpha, alpha)
        alpha = alpha + self.end_transitions.unsqueeze(0)
        return torch.logsumexp(alpha, dim=1)

    def decode(self, emissions, mask):
        B, T, C = emissions.shape
        viterbi = emissions[:, 0] + self.start_transitions.unsqueeze(0)
        backpointers = []
        for t in range(1, T):
            trans = self.transitions.unsqueeze(0)
            scores = viterbi.unsqueeze(1) + trans
            best_scores, best_ids = scores.max(dim=2)
            emit = emissions[:, t]
            new_viterbi = best_scores + emit
            viterbi = torch.where(mask[:, t].unsqueeze(1), new_viterbi, viterbi)
            backpointers.append(best_ids)
        viterbi = viterbi + self.end_transitions.unsqueeze(0)
        lengths = mask.long().sum(dim=1)
        best_paths = []
        for b in range(B):
            seq_len = lengths[b].item()
            best_last = viterbi[b].argmax().item()
            path = [best_last]
            for t in range(seq_len - 2, -1, -1):
                best_last = backpointers[t][b, best_last].item()
                path.append(best_last)
            path.reverse()
            path = path + [0] * (T - seq_len)
            best_paths.append(path)
        return best_paths


class LegalNER_Enc(nn.Module):
    """Encoder + CRF for NER. No BiLSTM — encoder is already bidirectional."""
    def __init__(self, encoder, num_tags, dropout=0.3):
        super().__init__()
        self.encoder = encoder
        self.num_tags = num_tags
        d_model = encoder.cfg.d_model
        self.ner_dropout = nn.Dropout(dropout)
        self.ner_head = nn.Linear(d_model, num_tags)
        self.crf = CRF(num_tags)
        nn.init.normal_(self.ner_head.weight, std=0.02)
        nn.init.zeros_(self.ner_head.bias)

    def forward(self, input_ids, labels=None, attention_mask=None):
        hidden = self.encoder(input_ids, attention_mask=attention_mask)
        emissions = self.ner_head(self.ner_dropout(hidden))
        out = {"logits": emissions}
        if labels is not None:
            crf_mask = (labels != -100)
            safe_labels = labels.clone()
            safe_labels[~crf_mask] = 0
            out["loss"] = self.crf(emissions, safe_labels, crf_mask)
        return out

    def decode(self, input_ids, attention_mask=None):
        hidden = self.encoder(input_ids, attention_mask=attention_mask)
        emissions = self.ner_head(hidden)
        crf_mask = attention_mask.bool() if attention_mask is not None else \
            torch.ones(input_ids.shape, dtype=torch.bool, device=input_ids.device)
        return self.crf.decode(emissions, crf_mask)

print(f"LegalNER_Enc + CRF: encoder -> CRF -> {CFG.bio_num_tags} BIO tags (no BiLSTM needed)")

LegalNER_Enc + CRF: encoder -> CRF -> 25 BIO tags (no BiLSTM needed)


## 8 — Load Encoder Weights from Enc-Dec Checkpoint

In [ ]:
def load_encoder_from_encdec(cfg, ckpt_path, device):
    encoder = LegalEncoder(cfg)
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state = ckpt.get("model", ckpt)

    # Extract only encoder keys: src_emb, enc_blocks, enc_norm
    enc_state = {}
    for k, v in state.items():
        if k.startswith(("src_emb", "enc_blocks", "enc_norm")):
            enc_state[k] = v

    missing, unexpected = encoder.load_state_dict(enc_state, strict=False)
    if missing:
        # rope buffers are non-persistent, that's fine
        real_missing = [m for m in missing if "rope" not in m]
        if real_missing:
            print(f"WARNING: Missing keys: {real_missing}")

    encoder = encoder.to(device)
    n_loaded = len(enc_state)
    print(f"Encoder loaded from {os.path.basename(ckpt_path)}: {n_loaded} tensors")
    print(f"  Params: {sum(p.numel() for p in encoder.parameters()):,}")
    return encoder

print("Encoder loader defined")

Encoder loader defined


## 9 — NER Dataset (Sliding Windows + BIO Tags)

In [ ]:
def assign_bio_tags(char_spans, gold_annotations, tag2id):
    n_tokens = len(char_spans)
    tags = [O_TAG] * n_tokens
    golds = sorted(gold_annotations, key=lambda a: int(a[0]))
    for ann in golds:
        gs, ge, label = int(ann[0]), int(ann[1]), ann[2]
        if label not in LABELS:
            continue
        b_tag = tag2id.get(f"B-{label}", O_TAG)
        i_tag = tag2id.get(f"I-{label}", O_TAG)
        first = True
        for t_idx, (cs, ce) in enumerate(char_spans):
            if ce <= cs:
                continue
            if cs < ge and ce > gs:
                tags[t_idx] = b_tag if first else i_tag
                first = False
    return tags


class NERDataset(Dataset):
    def __init__(self, text_dict, gold_dict, tokenizer, cfg, tag2id):
        self.windows = []
        self.cfg = cfg
        for doc_id in tqdm(text_dict, desc="Building NER dataset"):
            text = get_doc_text(text_dict[doc_id])
            if not text.strip():
                continue
            all_ids, all_spans = encode_with_offsets(tokenizer, text)
            n = len(all_ids)
            gold_anns = gold_dict.get(doc_id, [])
            all_tags = assign_bio_tags(all_spans, gold_anns, tag2id)
            if n == 0:
                continue
            step = cfg.window_stride if cfg.window_stride > 0 else cfg.window_size
            starts = list(range(0, max(1, n - cfg.window_size + 1), step))
            if starts[-1] + cfg.window_size < n:
                starts.append(n - cfg.window_size)
            for w_idx, s in enumerate(starts):
                e = min(s + cfg.window_size, n)
                self.windows.append((all_ids[s:e], all_tags[s:e], doc_id, w_idx))
        print(f"  {len(self.windows)} windows from {len(text_dict)} docs")

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        w_ids, w_tags, doc_id, w_idx = self.windows[idx]
        L = self.cfg.window_size
        pad_len = L - len(w_ids)
        return {
            "input_ids": torch.tensor(w_ids + [0]*pad_len, dtype=torch.long),
            "labels": torch.tensor(w_tags + [-100]*pad_len, dtype=torch.long),
            "attention_mask": torch.tensor([1]*len(w_ids) + [0]*pad_len, dtype=torch.long),
        }

print("NERDataset defined")

NERDataset defined


In [ ]:
# Train/Val split (no oversampling)
all_text = {**fold1_text, **fold2_text}
all_gold = {**fold1_gold, **fold2_gold}
doc_ids = sorted(all_text.keys())
random.seed(CFG.seed)
random.shuffle(doc_ids)
split_idx = len(doc_ids) - 3
train_ids = doc_ids[:split_idx]
val_ids = doc_ids[split_idx:]
train_text = {d: all_text[d] for d in train_ids}
train_gold = {d: all_gold[d] for d in train_ids}
val_text = {d: all_text[d] for d in val_ids}
val_gold = {d: all_gold[d] for d in val_ids}
print(f"Train: {len(train_text)} docs, Val: {len(val_text)} docs")

train_ds = NERDataset(train_text, train_gold, tokenizer, CFG, tag2id)
val_ds   = NERDataset(val_text, val_gold, tokenizer, CFG, tag2id)

print(f"No oversampling — {len(train_ds.windows)} windows")

train_dl = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
val_dl   = DataLoader(val_ds, batch_size=CFG.batch_size, shuffle=False, num_workers=4, pin_memory=True)
print(f"Train: {len(train_ds)} windows, Val: {len(val_ds)} windows")

Train: 67 docs, Val: 3 docs


Building NER dataset:   0%|          | 0/67 [00:00<?, ?it/s]

  1531 windows from 67 docs


Building NER dataset:   0%|          | 0/3 [00:00<?, ?it/s]

  61 windows from 3 docs
No oversampling — 1531 windows
Train: 1531 windows, Val: 61 windows


## 10 — Training + Doc-Level Evaluation

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_cosine_mult(step, total_steps, warmup_steps):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    prog = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * prog))

TRIM_CHARS = set('()[]{},:;/\\\'\"-!?.')

def _trim_spans(doc_spans, text):
    trimmed = []
    for s_start, s_end, s_label in doc_spans:
        if s_start >= s_end:
            continue
        snippet = text[s_start:s_end]
        l = 0
        while l < len(snippet) and snippet[l] in TRIM_CHARS:
            l += 1
        r = len(snippet)
        while r > l and snippet[r-1] in TRIM_CHARS:
            r -= 1
        if r > l:
            trimmed.append([s_start + l, s_start + r, s_label])
    return trimmed

def tag_preds_to_char_spans(tag_preds, char_spans, text):
    doc_spans = []
    current_label = None
    span_start = None
    span_end = None
    for i, tag_id in enumerate(tag_preds):
        tag_name = id2tag.get(tag_id, "O")
        cs, ce = char_spans[i]
        if tag_name.startswith("B-"):
            if current_label is not None and span_start is not None:
                doc_spans.append([span_start, span_end, current_label])
            current_label = tag_name[2:]
            span_start = cs
            span_end = ce
        elif tag_name.startswith("I-"):
            if current_label == tag_name[2:]:
                span_end = max(span_end, ce) if span_end else ce
            else:
                if current_label is not None and span_start is not None:
                    doc_spans.append([span_start, span_end, current_label])
                current_label = tag_name[2:]
                span_start = cs
                span_end = ce
        else:
            if current_label is not None and span_start is not None:
                doc_spans.append([span_start, span_end, current_label])
            current_label = None
            span_start = None
            span_end = None
    if current_label is not None and span_start is not None:
        doc_spans.append([span_start, span_end, current_label])
    return _trim_spans(doc_spans, text)

@torch.no_grad()
def evaluate_ner_doc_level(model, text_dict, gold_dict, tokenizer, cfg, device=DEVICE):
    model.eval()
    pred_spans_all = defaultdict(set)
    gold_spans_all = defaultdict(set)
    total_loss = 0.0
    n_batches = 0

    for doc_id in text_dict:
        text = get_doc_text(text_dict[doc_id])
        if not text.strip():
            continue
        all_ids, all_char_spans = encode_with_offsets(tokenizer, text)
        n = len(all_ids)
        if n == 0:
            continue
        gold_anns = gold_dict.get(doc_id, [])
        full_tags = assign_bio_tags(all_char_spans, gold_anns, tag2id)

        step = cfg.window_stride if cfg.window_stride > 0 else cfg.window_size
        starts = list(range(0, max(1, n - cfg.window_size + 1), step))
        if starts[-1] + cfg.window_size < n:
            starts.append(max(0, n - cfg.window_size))

        token_logits = [None] * n
        token_dist = [-1] * n
        for s in starts:
            e = min(s + cfg.window_size, n)
            w_ids = all_ids[s:e]
            w_tags = full_tags[s:e]
            pad_len = cfg.window_size - len(w_ids)
            inp = torch.tensor([w_ids + [0]*pad_len], dtype=torch.long, device=device)
            labs = torch.tensor([w_tags + [-100]*pad_len], dtype=torch.long, device=device)
            amask = torch.tensor([[1]*len(w_ids) + [0]*pad_len], dtype=torch.long, device=device)
            with torch.amp.autocast('cuda', enabled=cfg.use_amp):
                out = model(inp, labels=labs, attention_mask=amask)
            total_loss += out["loss"].item()
            n_batches += 1
            logits = out["logits"][0, :len(w_ids), :]
            for i_w in range(len(w_ids)):
                g = s + i_w
                dist = min(i_w, len(w_ids) - 1 - i_w)
                if dist > token_dist[g]:
                    token_dist[g] = dist
                    token_logits[g] = logits[i_w]

        emissions_list = []
        for i in range(n):
            emissions_list.append(
                token_logits[i] if token_logits[i] is not None
                else torch.zeros(model.num_tags, device=device)
            )
        emissions = torch.stack(emissions_list).unsqueeze(0)
        crf_mask = torch.ones(1, n, dtype=torch.bool, device=device)
        tag_preds = model.crf.decode(emissions, crf_mask)[0]

        pred_char = tag_preds_to_char_spans(tag_preds, all_char_spans, text)
        for ps, pe, pl in pred_char:
            pred_spans_all[pl].add((doc_id, ps, pe))
        for ann in gold_anns:
            gs, ge, label = int(ann[0]), int(ann[1]), ann[2]
            if label in LABELS:
                gold_spans_all[label].add((doc_id, gs, ge))

    per_label = {}
    for label in LABELS:
        ps = pred_spans_all.get(label, set())
        gs = gold_spans_all.get(label, set())
        tp = len(ps & gs)
        fp = len(ps - gs)
        fn = len(gs - ps)
        p = tp / (tp + fp) if tp + fp > 0 else 0
        r = tp / (tp + fn) if tp + fn > 0 else 0
        per_label[label] = 2 * p * r / (p + r) if p + r > 0 else 0
    macro_f1 = sum(per_label.values()) / len(LABELS)
    return total_loss / max(n_batches, 1), macro_f1, per_label


def train_ner(model, train_dl, val_text, val_gold, cfg, exp_name, device=DEVICE, patience=7):
    set_seed(cfg.seed)
    model = model.to(device)

    backbone_params = [(n, p) for n, p in model.named_parameters()
                       if "encoder" in n and p.requires_grad]
    head_params = [(n, p) for n, p in model.named_parameters()
                   if "encoder" not in n and p.requires_grad]
    head_lr = getattr(cfg, 'head_lr', 5e-4)

    param_groups = []
    if backbone_params:
        param_groups.append({"params": [p for _, p in backbone_params], "lr": cfg.lr})
    if head_params:
        param_groups.append({"params": [p for _, p in head_params], "lr": head_lr})

    n_bb = sum(p.numel() for _, p in backbone_params)
    n_hd = sum(p.numel() for _, p in head_params)
    print(f"Encoder: {n_bb:,} (lr={cfg.lr:.1e}) | Head: {n_hd:,} (lr={head_lr:.1e})")

    optimizer = torch.optim.AdamW(param_groups, weight_decay=cfg.weight_decay)
    base_lrs = [pg["lr"] for pg in optimizer.param_groups]
    scaler = torch.amp.GradScaler('cuda', enabled=cfg.use_amp)
    total_steps = len(train_dl) * cfg.num_epochs // cfg.grad_accum

    best_f1 = 0.0
    best_state = None
    global_step = 0
    no_improve = 0
    all_trainable = [p for p in model.parameters() if p.requires_grad]

    for epoch in range(cfg.num_epochs):
        model.train()
        running_loss = 0.0
        t0 = time.time()
        for batch_idx, batch in enumerate(train_dl):
            ids = batch["input_ids"].to(device)
            labs = batch["labels"].to(device)
            amask = batch["attention_mask"].to(device)

            with torch.amp.autocast('cuda', enabled=cfg.use_amp):
                out = model(ids, labels=labs, attention_mask=amask)
                loss = out["loss"] / cfg.grad_accum

            scaler.scale(loss).backward()
            running_loss += loss.item()

            if (batch_idx + 1) % cfg.grad_accum == 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(all_trainable, cfg.max_grad_norm)
                mult = get_cosine_mult(global_step, total_steps, cfg.warmup_steps)
                for pg, blr in zip(optimizer.param_groups, base_lrs):
                    pg["lr"] = max(cfg.min_lr, blr * mult)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1

            if (batch_idx + 1) % (50 * cfg.grad_accum) == 0:
                print(f"  [{exp_name}] Ep {epoch+1} step {batch_idx+1}/{len(train_dl)} "
                      f"| loss {running_loss/(batch_idx+1):.4f} | {time.time()-t0:.0f}s")

        val_loss, macro_f1, per_label = evaluate_ner_doc_level(
            model, val_text, val_gold, tokenizer, cfg, device)
        print(f"  [{exp_name}] Epoch {epoch+1} | val_loss={val_loss:.4f} | DOC macro_f1={macro_f1:.4f}")
        for lab, f1 in sorted(per_label.items(), key=lambda x: -x[1]):
            print(f"    {lab:30s} F1={f1:.4f}")

        if macro_f1 > best_f1:
            best_f1 = macro_f1
            best_state = copy.deepcopy(model.state_dict())
            torch.save({"model": best_state, "f1": best_f1, "epoch": epoch+1},
                       CKPT_DIR / f"{exp_name}_best.pt")
            print(f"  New best DOC F1={best_f1:.4f}")
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"  Training done. Best DOC macro-F1: {best_f1:.4f}")
    return model

print("Training + doc-level eval defined")

Training + doc-level eval defined


## 11 — Prediction & Submission

In [ ]:
@torch.no_grad()
def predict_test(model, text_dict, tokenizer, cfg, device=DEVICE):
    model.eval()
    predictions = {}
    for doc_id in tqdm(text_dict, desc="Predicting"):
        text = get_doc_text(text_dict[doc_id])
        if not text.strip():
            predictions[doc_id] = []
            continue
        all_ids, all_char_spans = encode_with_offsets(tokenizer, text)
        n = len(all_ids)
        if n == 0:
            predictions[doc_id] = []
            continue

        step = cfg.window_stride if cfg.window_stride > 0 else cfg.window_size
        starts = list(range(0, max(1, n - cfg.window_size + 1), step))
        if starts[-1] + cfg.window_size < n:
            starts.append(max(0, n - cfg.window_size))

        token_logits = [None] * n
        token_dist = [-1] * n
        for s in starts:
            e = min(s + cfg.window_size, n)
            w_ids = all_ids[s:e]
            pad_len = cfg.window_size - len(w_ids)
            inp = torch.tensor([w_ids + [0]*pad_len], dtype=torch.long, device=device)
            amask = torch.tensor([[1]*len(w_ids) + [0]*pad_len], dtype=torch.long, device=device)
            with torch.amp.autocast('cuda', enabled=cfg.use_amp):
                out = model(inp, attention_mask=amask)
            logits = out["logits"][0, :len(w_ids), :]
            for i_w in range(len(w_ids)):
                g = s + i_w
                dist = min(i_w, len(w_ids) - 1 - i_w)
                if dist > token_dist[g]:
                    token_dist[g] = dist
                    token_logits[g] = logits[i_w]

        emissions_list = [
            token_logits[i] if token_logits[i] is not None
            else torch.zeros(model.num_tags, device=device)
            for i in range(n)
        ]
        emissions = torch.stack(emissions_list).unsqueeze(0)
        crf_mask = torch.ones(1, n, dtype=torch.bool, device=device)
        tag_preds = model.crf.decode(emissions, crf_mask)[0]
        predictions[doc_id] = tag_preds_to_char_spans(tag_preds, all_char_spans, text)
    return predictions


def save_submission(predictions, exp_name):
    pred_path = SUB_DIR / f"prediction_{exp_name}.json"
    zip_path  = SUB_DIR / f"submission_{exp_name}.zip"
    with open(pred_path, "w") as f:
        json.dump(predictions, f, ensure_ascii=False)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(pred_path, "prediction.json")
    print(f"  {exp_name}: {len(predictions)} docs, "
          f"{sum(len(v) for v in predictions.values())} entities -> {zip_path.name}")
    return zip_path

print("Prediction & submission defined")

Prediction & submission defined


## 12 — Run Experiment

In [ ]:
set_seed(CFG.seed)
encoder = load_encoder_from_encdec(CFG, PHASE1_CKPT, DEVICE)
model_enc = LegalNER_Enc(encoder, CFG.bio_num_tags, dropout=0.3)

for p in model_enc.parameters():
    p.requires_grad = True

cfg_enc = copy.copy(CFG)
cfg_enc.lr = 1e-5         # encoder LR — can be a bit higher since encoder is smaller
cfg_enc.head_lr = 5e-4    # CRF + classifier
cfg_enc.warmup_steps = 200
cfg_enc.num_epochs = 25
cfg_enc.min_lr = 1e-7

model_enc = train_ner(model_enc, train_dl, val_text, val_gold, cfg_enc,
                      "exp_encoder_bio", patience=7)

preds = predict_test(model_enc, fold3_text, tokenizer, CFG)
save_submission(preds, "exp_encoder_bio")

del encoder
torch.cuda.empty_cache()

Encoder loaded from bestencdec.pt: 47 tensors
  Params: 48,339,840
Encoder: 48,339,840 (lr=1.0e-05) | Head: 16,700 (lr=5.0e-04)
  [exp_encoder_bio] Ep 1 step 100/191 | loss 801.2968 | 51s
  [exp_encoder_bio] Epoch 1 | val_loss=623.4530 | DOC macro_f1=0.0000
    APP                            F1=0.0000
    RESP                           F1=0.0000
    A.COUNSEL                      F1=0.0000
    R.COUNSEL                      F1=0.0000
    JUDGE                          F1=0.0000
    WIT                            F1=0.0000
    AUTH                           F1=0.0000
    COURT                          F1=0.0000
    STAT                           F1=0.0000
    PREC                           F1=0.0000
    DATE                           F1=0.0000
    CASENO                         F1=0.0000
  [exp_encoder_bio] Ep 2 step 100/191 | loss 167.6558 | 50s
  [exp_encoder_bio] Epoch 2 | val_loss=129.7183 | DOC macro_f1=0.0318
    DATE                           F1=0.3820
    APP                    

Predicting:   0%|          | 0/35 [00:00<?, ?it/s]

  exp_encoder_bio: 35 docs, 5193 entities -> submission_exp_encoder_bio.zip


In [ ]:
# Save results to Drive
drive_out = Path("/content/drive/MyDrive/phase2_ner_results")
drive_out.mkdir(parents=True, exist_ok=True)

for zf in sorted(SUB_DIR.glob("submission_*.zip")):
    shutil.copy(zf, drive_out / zf.name)
    print(f"Saved to Drive: {drive_out / zf.name}")

best_ckpt = CKPT_DIR / "exp_encoder_bio_best.pt"
if best_ckpt.exists():
    shutil.copy(best_ckpt, drive_out / best_ckpt.name)
    print(f"Saved checkpoint to Drive: {drive_out / best_ckpt.name}")

from google.colab import files
for zf in sorted(SUB_DIR.glob("submission_*.zip")):
    files.download(str(zf))
print("Done!")

Saved to Drive: /content/drive/MyDrive/phase2_ner_results/submission_exp_encoder_bio.zip
Saved checkpoint to Drive: /content/drive/MyDrive/phase2_ner_results/exp_encoder_bio_best.pt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done!
